In [0]:
df = spark.read.table("bronze_mydata")

In [0]:
from pyspark.sql import functions as F

In [0]:
for col_name in df.columns:
    df = df.withColumnRenamed(col_name, col_name.replace(" ", "_"))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6583570193746885>, line 1
----> 1 for col_name in df.columns:
      2     df = df.withColumnRenamed(col_name, col_name.replace(" ", "_"))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:309, in DataFrame.columns(self)
    307 @property
    308 def columns(self) -> List[str]:
--> 309     return [field.name for field in self._schema.fields]

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1977, in DataFrame._schema(self)
   1975 if self._cached_schema is None:
   1976     query = self._plan.to_proto(self._session.client)
-> 1977     self._cached_schema = self._session.client.schema(query)
   1978     try:
   1979         self._cached_schema_serialized = CPickleSerializer().dumps(self._schema)

File /databricks/python/lib/python3.12/site-packages/pys

In [0]:
df = df.withColumn("DATE_OCC_DT", F.to_date(F.col("DATE_OCC"))) \
       .withColumn("Date_Rptd_DT", F.to_date(F.col("Date_Rptd"))) \
       .withColumn("Year", F.year("DATE_OCC_DT")) \
       .withColumn("Month", F.month("DATE_OCC_DT")) \
       .withColumn("DayOfWeek", F.date_format("DATE_OCC_DT", "EEEE")) \
       .withColumn("Hour", (F.col("TIME_OCC") / 100).cast("int")) \
       .withColumn("IsWeekend", F.when(F.dayofweek(F.col("DATE_OCC_DT")).isin(1, 7), 1).otherwise(0))

In [0]:
df = df.withColumn("Reporting_Delay", F.datediff(F.col("Date_Rptd"), F.col("DATE_OCC_DT"))) \
       .withColumn("Reporting_Delay", F.when(F.col("Reporting_Delay") < 0, 0).otherwise(F.col("Reporting_Delay")))

In [0]:
df = df.withColumn("Vict_Age_Clean", 
                  F.when((F.col("Vict_Age") < 1) | (F.col("Vict_Age") > 100), F.lit(None))
                  .otherwise(F.col("Vict_Age")))

In [0]:
df = df.withColumn("Vict_Sex_Clean", 
                  F.when(F.col("Vict_Sex") == "M", "Male")
                  .when(F.col("Vict_Sex") == "F", "Female")
                  .otherwise("Unknown"))

In [0]:
df = df.withColumn("Has_Weapon", 
                   F.when(F.col("Weapon_Used_Cd").isNotNull(), 1)
                   .otherwise(0))

In [0]:
df = df.withColumn("Reporting_Delay", F.datediff(F.col("Date_Rptd"), F.col("DATE_OCC")))

In [0]:
df = df.filter(F.col("Crm_Cd_Desc").isNotNull())

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("silver_mydata")
print("Success! Silver Layer 'silver_mydata' created.")